# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_wavelength
from functions import laser_set_standard, laser_get_standard
import snspd
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260414-35992-qcodes.log
Experiment loaded. Last ID no: 476


In [2]:
import functions
import importlib
importlib.reload(functions)
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_wavelength
from functions import laser_set_standard, laser_get_standard

# Instruments

In [2]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("mso5", revive_instance=True)
pm100usb = station.load_instrument("pm100usb", revive_instance=True)
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

2026-04-14 16:15:56,347 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [yoko_measure(GS200_Monitor)] Snapshot: Could not update parameter: trigger
2026-04-14 16:15:56,362 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [yoko(GS200)] Snapshot: Could not update parameter: voltage_range
2026-04-14 16:15:56,366 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [yoko(GS200)] Snapshot: Could not update parameter: current_range
2026-04-14 16:15:56,368 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [yoko(GS200)] Snapshot: Could not update parameter: range
2026-04-14 16:15:56,372 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [yoko(GS200)] Snapshot: Could not update parameter: voltage
2026-04-14 16:15:56,374 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [y

Connected to: Agilent Technologies 34410A (serial:MY47027892, firmware:2.35-2.35-0.09-46-09) in 0.11s
Connected to: YOKOGAWA GS210 (serial:91T928105, firmware:2.02) in 0.03s
Connected to: PurePhotonic PPCL550 (serial:PP70AJ005, firmware:PV 2.0.0:HW 3.0.0:FW 7.0.0:AS C1:OT 1.0.0) in 1.56s


2026-04-14 16:15:58,986 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\MSO5.py:5: QCoDeSDeprecationWarning: The `qcodes.instrument.base` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.instrument.base import InstrumentBase

2026-04-14 16:16:00,509 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument ¦ get_idn ¦ 113 ¦ [pm100usb(Thorlabs_PM100USB)] Error getting or interpreting *IDN?: ''
Traceback (most recent call last):
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\instrument.py", line 100, in get_idn
    idstr = self.ask("*IDN?")
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\instrument.py", line 445, in ask
    raise e
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\instrument.py", line 439, in ask
    answer = self.ask_raw(cmd)
  File "C:\Users\QNL\anaconda3\env

Connected to: Keithley instruments 2230G-30-1 (serial:9010428, firmware:1.16-1.04) in 0.15s


# Light Counts vs Wavelength

In [17]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

print(params.v_scale_counts)
print(params.h_time_counts)
print(params.h_pos_counts)

wavelength_range = np.arange(1528e-9, 1565e-9, 1e-9) # wavelength range of PPCL550
# wavelength_range = np.arange(1528e-9, 1529e-9, 1e-9) # testing
v_attenuator = 4.7 # from SNSPD4-2_Calibration.ipynb ID 457

# Set oscilloscope vertical and horizontal parameters 
osc_set_standard(MS, v_scale=params.v_scale_counts, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
osc_check_standard(MS)

if MS.channels[0].clipping(): 
    print('Error: Clipping')

# Set attenuator voltage
p_att.write(f'VOLT {v_attenuator}')

# Set standard laser parameters 
laser_set_standard(laser, wavelength_range[0], 7)
laser_get_standard(laser)

# Set current 
yoko.current(12e-6)

snspd_counts_vs_wavelength(MS, dmm, yoko, p_att, laser, device_name=params.device_1_name, n_captures=10, interval=1, wavelength_range=wavelength_range, station=station)
# at the time of the run SNSPD4_1_Counts_vs_Attenuation had been deleted, copied parameters from that file on GitHub


############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False
0.15
0.1
0
MANUAL
RECORDLENGTH
625000000.0
10.0
0.01
0.0
0.15
50.0
1000000000.0
0.0
0.0
0.0
1.0
0
0.0
Power: 7.0
Frequency coarse: 196.1992THz
Wavelength (calculated) is 1528.0004097876035nm
update station
Starting experimental run with id: 472. 
472
Laser enable status: False
Power: 7.0
Frequency coarse: 196.1992THz
Wavelength (calculated) is 1528.0004097876035nm


Laser enable status: True
This acquisition will take 10s
12 29
Laser enable status: False
Power: 7.0
Frequency coarse: 196.0709THz
Wavelength (calculated) is 1529.0002647001672nm


Laser enable status: True
This acquisition will take 10s
12 29
Laser enable status: False
Power: 7.0
Frequency coarse: 195.9427THz
Wavelength (calculated) is 1530.000648148668nm


Laser enable status: True
This acquisition will take 10s
12 30
Laser enable status: False
Power: 7.0
Frequency coarse: 195.8147THz
Wavelength (calculated) is 1531.0007777761323nm


Laser enable status: True
This acquisition will take 10s
12 30